# Actinium chelation workflow

This example binds macropa to Ac and samples ligand configurations for energy reference.

Basic schema from this paper:

https://pubs.acs.org/doi/full/10.1021/acs.inorgchem.0c02432

In [25]:
from architector import build_complex, convert_io_molecule, view_structures
import numpy as np
from tqdm import tqdm
from ase import units

In [26]:
macropa_out = build_complex(
    {'core':{'metal':'Ac','coreCN':10},
     'ligands':['macropa'],
     'parameters':{
      'assemble_method':'GFN-FF',
      'full_method':'GFN2-xTB',
      'xtb_solvent':'water',
      # New architector functions:
      'n_lig_conformers':3, # Gives ~10 ligand structures per symmetry conformers
      'n_complex_relax':10, # Relax all 10 ligand structures and put in solution pool
      'skip_duplicate_tests':True,
      'return_full_complex_class':True}}
)

In [27]:
view_structures(macropa_out)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [28]:
dota_out = build_complex(
    {'core':{'metal':'Ac','coreCN':8},
     'ligands':['dota-'],
     'parameters':{
      'assemble_method':'GFN-FF',
      'full_method':'GFN2-xTB',
      'xtb_solvent':'water',
      # New architector functions:
      'n_lig_conformers':3, # Gives ~10 ligand structures per symmetry conformers
      'n_complex_relax':10, # Relax all 10 ligand structures and put in solution pool
      'skip_duplicate_tests':True,
      'return_full_complex_class':True}}
)

In [29]:
view_structures(dota_out)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [30]:
complex_energies = np.array([macropa_out[key]['energy'] for key in macropa_out.keys()])
macropa_energy = complex_energies.min()
(complex_energies - complex_energies.min()) / (units.kcal/units.mol)

array([ 0.        ,  0.86788293,  1.04845899,  1.4416658 ,  1.4416921 ,
        1.45373502,  1.98909171,  2.0514892 ,  2.05944712,  2.86720839,
        2.8744582 ,  4.70081026,  4.94152999,  5.1355263 ,  5.20991406,
        5.41845961,  6.22170728,  6.68841054,  7.21257605,  7.76725035,
        7.79737369,  8.55150722,  8.55151395,  9.99945936, 10.43823103,
       11.06941388, 11.47343548, 13.58041363, 13.58528237, 17.60081844])

In [31]:
complex_energies = np.array([dota_out[key]['energy'] for key in dota_out.keys()])
dota_energy = complex_energies.min()
(complex_energies - complex_energies.min()) / (units.kcal/units.mol)

array([0.00000000e+00, 6.02636128e-03, 8.37784761e-03, 1.38800576e+00,
       3.19787479e+00, 3.19990555e+00, 3.20057307e+00, 3.20175847e+00,
       3.20329983e+00, 3.20575153e+00, 3.20781659e+00, 3.21026979e+00,
       3.21391702e+00, 3.21875434e+00, 3.21929926e+00, 3.22007638e+00,
       3.22049805e+00, 4.35959156e+00, 7.23425612e+01, 7.23454508e+01,
       7.23471201e+01, 7.23482049e+01, 7.23482534e+01, 7.23487442e+01,
       7.23508024e+01, 7.23511613e+01, 7.23514679e+01, 7.23516812e+01,
       7.23554626e+01, 7.23564453e+01, 7.23568944e+01, 7.23569249e+01,
       7.23570890e+01, 7.25645094e+01])

In [32]:
# Generate isolated ligand structures
from architector import CalcExecutor
from architector.io_obabel import generate_obmol_conformers
from architector.io_ptable import ligands_dict
from architector.io_align_mol import rmsd_group

In [34]:
lig = 'macropa'
confs, energies = generate_obmol_conformers(
    ligands_dict[lig]['smiles'], return_energies=True
)
sets, selected_confs = rmsd_group(
    confs,
    rmsd_type='simple',
    value_list=energies,
    return_new_conf_list=True
)
xtb_conformers = []
for conf in tqdm(selected_confs[0:10],total=10):
    out1 = CalcExecutor(conf,method='GFN2-xTB',xtb_solvent='water',
            relax=True,xtb_relax=True)
    if out1.successful:
        xtb_conformers.append(out1)
macropa_lig_energy = min([x.energy for x in xtb_conformers])
view_structures([x.mol for x in xtb_conformers])

..tot conformations = 11664
..tot confs tested = 3000
..below energy threshold = 1454


 20%|██        | 2/10 [00:24<01:38, 12.32s/it]


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [35]:
lig = 'dota-'
confs, energies = generate_obmol_conformers(
    ligands_dict[lig]['smiles'], return_energies=True
)
sets, selected_confs = rmsd_group(
    confs,
    rmsd_type='simple',
    value_list=energies,
    return_new_conf_list=True
)
xtb_conformers = []
for conf in tqdm(selected_confs[0:10],total=10):
    out1 = CalcExecutor(conf,method='GFN2-xTB',xtb_solvent='water',
            relax=True,xtb_relax=True)
    if out1.successful:
        xtb_conformers.append(out1)
dota_lig_energy = min([x.energy for x in xtb_conformers])
view_structures([x.mol for x in xtb_conformers])

..tot conformations = 331776
..tot confs tested = 3000
..below energy threshold = 335


 30%|███       | 3/10 [00:09<00:21,  3.02s/it]


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [38]:
reaction_energy = macropa_energy + dota_lig_energy - dota_energy - macropa_lig_energy
print(reaction_energy/(units.kcal/units.mol))

-18.411817619430963
